In [1]:
path = r"C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\"
# path = r"C:\Users\samcl\Massachusetts Institute of Technology\Truck Parking Capstone - Truck Stop Finder 🚚⛽\\"

In [2]:
import folium
import geopandas as gpd
import h3

# ---- 1) Read the Census shapefile ----
# Point this to the .shp inside your cb_2024_us_state_500k folder
# shp_path = r"cb_2024_us_state_500k/cb_2024_us_state_500k.shp"
states = gpd.read_file(
    path + r"5. Source & Refrence Files\1. Shapefiles\cb_2024_us_state_500k\cb_2024_us_state_500k.shp")

fl = states.loc[states["STUSPS"] == "FL"].to_crs(epsg=4326)
fl_geom = fl.iloc[0].geometry


# ---- 2) Shapely -> H3Shape (LatLngPoly / list of LatLngPoly) ----
def shapely_to_latlngpoly(geom):
    """
    Converts shapely Polygon/MultiPolygon to H3 LatLngPoly / list[LatLngPoly].
    Note: H3 wants (lat, lng) tuples.
    """
    from shapely.geometry import Polygon

    def poly_to_latlngpoly(poly: Polygon):
        outer = [(y, x) for (x, y) in poly.exterior.coords]  # (lat, lon)
        holes = []
        for ring in poly.interiors:
            holes.append([(y, x) for (x, y) in ring.coords])  # (lat, lon)
        return h3.LatLngPoly(outer, *holes)

    if geom.geom_type == "Polygon":
        return poly_to_latlngpoly(geom)
    elif geom.geom_type == "MultiPolygon":
        return [poly_to_latlngpoly(p) for p in geom.geoms]
    else:
        raise ValueError(f"Unsupported geometry type: {geom.geom_type}")


# ---- 3) Polyfill Florida ----
resolution = 4

h3shape = shapely_to_latlngpoly(fl_geom)

# polygon_to_cells can take a single LatLngPoly.
# For MultiPolygon we union the cells from each polygon.
if isinstance(h3shape, list):
    fl_cells = set()
    for s in h3shape:
        fl_cells |= set(h3.polygon_to_cells(s, resolution))
else:
    fl_cells = set(h3.polygon_to_cells(h3shape, resolution))

print("Number of H3 cells:", len(fl_cells))

# ---- 4) Plot on Folium ----
m = folium.Map(location=[27.8, -81.7], zoom_start=6)

for cell in fl_cells:
    boundary = h3.cell_to_boundary(cell)  # [(lat, lon), ...]
    folium.Polygon(
        locations=boundary,
        weight=0.4,
        fill=False,
        opacity=0.7
    ).add_to(m)

# m


Number of H3 cells: 93


In [7]:
import random
import pandas as pd
import folium
import h3

# ----------------------------
# Inputs you already have:
# fl_cells = set([...])  # H3 cell IDs for Florida
# ----------------------------

# 1) Randomly assign colors (set seed for reproducibility)
random.seed(42)
palette = ["red", "yellow", "green"]

cell_color = {cell: random.choice(palette) for cell in fl_cells}

# 2) Build table: polygon id, centroid, color
rows = []
for cell in fl_cells:
    lat, lon = h3.cell_to_latlng(cell)  # centroid (lat, lon)
    rows.append({
        "h3_id": cell,
        "centroid_lat": lat,
        "centroid_lon": lon,
        "color": cell_color[cell]
    })

df = pd.DataFrame(rows)

# Optional: sort for readability
df = df.sort_values(["color", "h3_id"]).reset_index(drop=True)

# 3) Export to Excel
out_path = "florida_h3_random_colors.xlsx"
df.to_excel(out_path, index=False)
print("Saved:", out_path)

# 4) Plot on Folium map with fill color
m = folium.Map(location=[27.8, -81.7], zoom_start=6, tiles="cartodbvoyager")

for cell in fl_cells:
    boundary = h3.cell_to_boundary(cell)  # [(lat, lon), ...]
    folium.Polygon(
        locations=boundary,
        color="black",  # border color
        weight=0.3,
        fill=True,
        fill_color=cell_color[cell],
        fill_opacity=0.35,
        opacity=0.5,
        tooltip=f"{cell} | {cell_color[cell]}"
    ).add_to(m)

# m


Saved: florida_h3_random_colors.xlsx


In [8]:
m.save("FL_h3.html")